# 03 — Deployment Walkthrough: Azure App Service (Native), with Docker as an Alternative

**Correction from an earlier version of this notebook:** the real `upload-service` deployment does
**not** use Docker. `FULL_ARCHITECTURE.md` section 11 and `DEPLOYMENT_REQUIREMENTS.md` both confirm the
real deployment target is **Azure App Service's native Python runtime**, running `uvicorn`/`gunicorn`
directly against `app.py`, configured entirely through Application Settings (environment variables).
This notebook now walks through *that* real deployment path first, and keeps a Docker
containerization walkthrough afterward — clearly labeled as **an alternative you could propose**,
not a description of what actually runs in production. See chapter 08
(`08-deployment-azure-app-service.md`) for the full narrative version of both.

Like the original version of this notebook, this is deliberately **markdown-heavy, not code-heavy**.
Its subject (`az` CLI commands, startup configuration) is fundamentally about shell commands, not
something meaningfully "run" inside a notebook kernel against real Azure resources — doing that here
would require a real Azure subscription and break this course's "runs offline" rule for notebooks. The
two code cells print the *real* startup command / Application Settings inventory, and the *proposed*
Dockerfile, as safe, introspective Python — nothing here calls Docker, `az`, or any live Azure API.


## 1. The real deployment: App Service's native Python runtime, no container

There is no `Dockerfile`, no container registry, and no `docker build`/`docker push` step in the real
deployment path. App Service needs an explicit **startup command** instead — the native-runtime
equivalent of a Dockerfile's `CMD` line — and every configuration value `app.py` reads via
`os.environ` comes from **Application Settings**, not a baked-in image layer (chapter 07).


In [1]:
# Safe, introspective only -- prints the real startup command and the real Application Settings
# inventory this service depends on (from DEPLOYMENT_REQUIREMENTS.md / FULL_ARCHITECTURE.md section 11a).
# Nothing here calls Docker, the az CLI, or a live Azure API.

startup_command = (
    "gunicorn --bind 0.0.0.0:8000 -k uvicorn.workers.UvicornWorker app:app"
)

runtime_stack = "PYTHON|3.11"

application_settings_preexisting = [
    "INGEST_API", "IDENTITY_CLIENT_ID", "MICROSOFT_PROVIDER_AUTHENTICATION_SECRET",
    "AZURE_TENANT_ID", "IDENTITY_SCOPE", "IDENTITY_REDIRECT_URI",
    "DATABASE_DRIVER", "DATABASE_SERVER", "DATABASE_NAME",
    "ENVIRONMENT", "IS_LOCAL_MACHINE", "BUILD_SOURCE", "BUILD_VERSION", "PORT",
]

application_settings_iwpb = [
    "SMTP_HOST", "SMTP_PORT", "SMTP_USERNAME", "SMTP_PASSWORD", "SMTP_USE_TLS",
    "SMTP_DISTRIBUTION_MAILBOX", "UPLOADER_APP_URL",
    "AZURE_STORAGE_ACCOUNT_URL", "AZURE_STORAGE_CONNECTION_STRING",
    "PENDING_UPLOAD_CONTAINER", "IWPB_MAINTENANCE_INTERVAL_SECONDS", "INGEST_API_SCOPE",
]

print("Real startup command (App Service > Configuration > General settings > Startup Command):")
print(f"  {startup_command}")
print()
print(f"Real runtime stack (App Service > Configuration > General settings): {runtime_stack}")
print()
print(f"Application Settings, pre-existing ({len(application_settings_preexisting)}):")
for name in application_settings_preexisting:
    print(f"  - {name}")
print()
print(f"Application Settings, added for the IWPB feature ({len(application_settings_iwpb)}):")
for name in application_settings_iwpb:
    print(f"  - {name}")


Real startup command (App Service > Configuration > General settings > Startup Command):
  gunicorn --bind 0.0.0.0:8000 -k uvicorn.workers.UvicornWorker app:app

Real runtime stack (App Service > Configuration > General settings): PYTHON|3.11

Application Settings, pre-existing (14):
  - INGEST_API
  - IDENTITY_CLIENT_ID
  - MICROSOFT_PROVIDER_AUTHENTICATION_SECRET
  - AZURE_TENANT_ID
  - IDENTITY_SCOPE
  - IDENTITY_REDIRECT_URI
  - DATABASE_DRIVER
  - DATABASE_SERVER
  - DATABASE_NAME
  - ENVIRONMENT
  - IS_LOCAL_MACHINE
  - BUILD_SOURCE
  - BUILD_VERSION
  - PORT

Application Settings, added for the IWPB feature (12):
  - SMTP_HOST
  - SMTP_PORT
  - SMTP_USERNAME
  - SMTP_PASSWORD
  - SMTP_USE_TLS
  - SMTP_DISTRIBUTION_MAILBOX
  - UPLOADER_APP_URL
  - AZURE_STORAGE_ACCOUNT_URL
  - AZURE_STORAGE_CONNECTION_STRING
  - PENDING_UPLOAD_CONTAINER
  - IWPB_MAINTENANCE_INTERVAL_SECONDS
  - INGEST_API_SCOPE


## 2. Setting the real deployment up, via `az` CLI (reference only, not executed here)

```bash
# One-time: point App Service at the native Python runtime stack (no image to build or push)
az webapp config set \
  --name upload-service \
  --resource-group my-rg \
  --linux-fx-version "PYTHON|3.11"

# Tell App Service how to start the app -- the native-runtime equivalent of a Dockerfile CMD
az webapp config set \
  --name upload-service \
  --resource-group my-rg \
  --startup-file "gunicorn --bind 0.0.0.0:8000 -k uvicorn.workers.UvicornWorker app:app"

# Set Application Settings (env vars) -- chapter 07. A real deployment sets every variable printed
# above; this is illustrative of the mechanism, not the full command.
az webapp config appsettings set \
  --name upload-service \
  --resource-group my-rg \
  --settings INGEST_API="https://ingest-api.example/" ENVIRONMENT="PROD" \
             SMTP_HOST="smtp.hsbc.internal" AZURE_STORAGE_ACCOUNT_URL="https://acct.blob.core.windows.net"

# Code deploy (zip deploy is a common choice for the native-runtime path -- no registry involved)
az webapp deploy \
  --name upload-service \
  --resource-group my-rg \
  --src-path upload-service.zip \
  --type zip
```

**Deployment slots** work the same way as they would for a containerized deployment — deploy to a
`staging` slot first, smoke-test, then swap with zero downtime:

```bash
az webapp deployment slot create --name upload-service --resource-group my-rg --slot staging
az webapp deploy --name upload-service --resource-group my-rg --slot staging --src-path upload-service.zip --type zip
# After smoke-testing the staging slot's own URL:
az webapp deployment slot swap --name upload-service --resource-group my-rg --slot staging --target-slot production
```


## 3. Health checks and the warm-up gate (real, already shipped)

```bash
# Tail App Service logs
az webapp log tail --name upload-service --resource-group my-rg

# The real health endpoint -- returns build version/environment/uptime for Azure health probes
curl https://upload-service.azurewebsites.net/health
```

`app.py`'s `read_root()` deliberately returns a plain "still loading" response instead of triggering an
OAuth redirect for the **first 180 seconds** after process start (`time.time() - start_time < 180`) —
this is what keeps a health probe hitting `/` right after a cold start or a slot swap from being
redirected into an auth flow it can't complete. `/docs` (Swagger UI) is also explicitly disabled when
`ENVIRONMENT == "PROD"` — both are real, confirmed, already-shipped operational patterns (chapter 06),
not proposals.


## 4. Scaling: what's safe, and the one real caveat (chapter 06)

The App Service Plan behind this service can scale to multiple instances. Azure Blob Storage (IWPB
staging) and Azure SQL (`approver_mapping`, `iwpb_document_workflow`) are both shared and
multi-instance-safe by design — any instance can serve any request against them. Two things are
**not** shared, and are worth remembering as a real operational caveat rather than a hypothetical:

- `memory_cache` (the bearer-token cache) is a **per-process, in-memory dict** — a session's cached
  token only exists on the instance that handled that session's sign-in.
- `_iwpb_maintenance_loop` (the hourly reminder/auto-removal/expiry-purge sweep) runs **independently
  in every instance** — not a singleton. Worst case under scale-out is a duplicate reminder email in
  the same window, never data corruption, since every write in the sweep is an idempotent SQL `UPDATE`.

See chapter 06 for the concrete fixes (a shared cache / session affinity; an Azure Functions Timer
trigger or a distributed lock) if either of these became a real, reported problem under scale-out.


## 5. Alternative: containerizing this service (a proposal, not reality)

Everything below is **transferable Docker knowledge you could propose for this service** — kept
here deliberately, clearly labeled, rather than deleted outright. If asked "how would you containerize
this," this is a legitimate, competent answer to give, framed correctly: *"the real system runs on
App Service's native Python runtime with no container — but here's how I'd containerize it and why it
might be worth doing."* The multi-stage Dockerfile below is the same shape discussed in chapter 08.


In [2]:
# Displayed only, as a proposal -- not built, run, or pushed anywhere in this notebook.
# This mirrors chapter 08's proposed Dockerfile for upload-service, not anything the real
# deployment actually uses.

dockerfile_contents = r"""
# ---- Stage 1: install dependencies ----
FROM python:3.11-slim AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# ---- Stage 2: slim runtime image ----
FROM python:3.11-slim
WORKDIR /app

COPY --from=builder /root/.local /root/.local
ENV PATH=/root/.local/bin:$PATH

COPY . .

RUN useradd --create-home appuser
USER appuser

EXPOSE 8000

# gunicorn as a process manager, uvicorn.workers.UvicornWorker for ASGI/async support --
# the same production-appropriate choice discussed in chapter 08.
CMD ["gunicorn", "--bind", "0.0.0.0:8000", "-k", "uvicorn.workers.UvicornWorker", "app:app"]
"""

print(dockerfile_contents)
print(f"Proposed Dockerfile is {len(dockerfile_contents.splitlines())} lines.")



# ---- Stage 1: install dependencies ----
FROM python:3.11-slim AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# ---- Stage 2: slim runtime image ----
FROM python:3.11-slim
WORKDIR /app

COPY --from=builder /root/.local /root/.local
ENV PATH=/root/.local/bin:$PATH

COPY . .

RUN useradd --create-home appuser
USER appuser

EXPOSE 8000

# gunicorn as a process manager, uvicorn.workers.UvicornWorker for ASGI/async support --
# the same production-appropriate choice discussed in chapter 08.
CMD ["gunicorn", "--bind", "0.0.0.0:8000", "-k", "uvicorn.workers.UvicornWorker", "app:app"]

Proposed Dockerfile is 24 lines.


## 6. If this proposal were adopted: registry + Web App for Containers (still a proposal)

```bash
# Authenticate to the registry
az acr login --name myregistry

# Build, tag, push
docker build -t upload-service:1.0.0 .
docker tag upload-service:1.0.0 myregistry.azurecr.io/upload-service:1.0.0
docker push myregistry.azurecr.io/upload-service:1.0.0

# Switch App Service from the native runtime stack to Web App for Containers
az webapp config container set \
  --name upload-service \
  --resource-group my-rg \
  --container-image-name myregistry.azurecr.io/upload-service:1.0.0

az webapp config appsettings set \
  --name upload-service \
  --resource-group my-rg \
  --settings WEBSITES_PORT=8000
```

**What would actually need to change to adopt this:** the deployment mechanism
(`config container set` instead of `config set --startup-file` / zip deploy) and standing up a
registry — nothing in `app.py`'s reliance on `os.environ` for configuration changes, since Application
Settings are injected as environment variables identically whether the process is running inside a
container or directly on the native runtime. That's exactly what makes this a low-risk proposal to
make in an interview rather than a rewrite.


## Summary

| Step | Real (native App Service runtime) | Alternative (containerized — proposal only) |
|---|---|---|
| Configure runtime | `az webapp config set --linux-fx-version "PYTHON\|3.11"` | `docker build`, push to ACR |
| Startup | `az webapp config set --startup-file "gunicorn ..."` | Dockerfile `CMD` |
| Config | Application Settings (chapter 07) | Same Application Settings, injected into the container |
| Deploy | `az webapp deploy --type zip` | `az webapp config container set --container-image-name ...` |
| Zero-downtime release | `az webapp deployment slot ...` (identical either way) | `az webapp deployment slot ...` (identical either way) |
| Health check | `GET /health`, warm-up gate (real, chapter 06) | Same — unaffected by the runtime choice |
| Scaling caveats | `memory_cache`, `_iwpb_maintenance_loop` not shared across instances (chapter 06) | Same — unaffected by the runtime choice |

None of the `az`/`docker` commands above were executed by this notebook — they're reference material.
The only code that ran was the two safe, introspective cells in sections 1 and 5 (printing the real
startup configuration, and printing the proposed Dockerfile as a string).
